# Trabalho Prático – Estrutura de Dados II (DCA3702)

## Análise Estrutural de Redes Urbanas com OSMnx, NetworkX e Gephi

Este notebook descreve o trabalho prático associado aos conteúdos das ``Semanas 5 e 6`` da disciplina de **Estrutura de Dados II**, com foco em ``Hubs`` e ``Core Decomposition``.

A proposta é utilizar dados reais de redes urbanas extraídas do OpenStreetMap por meio da biblioteca [OSMnx](https://github.com/gboeing/osmnx), realizar análises estruturais com [NetworkX](https://networkx.org/en/) e produzir visualizações avançadas no [Gephi](https://gephi.org/).

## 1. Objetivo

O objetivo deste trabalho é aplicar conceitos de grafos em uma rede real, interpretando a malha viária de uma cidade ou bairro como um grafo.

Os alunos deverão identificar e analisar:

- nós centrais da rede, também chamados de ``hubs``;
- regiões estruturalmente densas por meio da decomposição ``k-core``;
- pontos críticos de conexão e fluxo usando métricas de centralidade;
- diferenças entre a visualização geográfica da rede e a visualização estrutural baseada em layouts de força.

> A proposta **não é apenas executar bibliotecas**, mas compreender como estruturas de dados modelam sistemas reais.

## 2. Problema norteador

Cada grupo deverá responder à seguinte questão:

> **Quais são os elementos estruturais mais importantes da malha viária analisada e como diferentes métricas de grafos, como grau, centralidade e k-core, ajudam a caracterizá-los?**

A análise deve ir além da apresentação de gráficos. Espera-se uma interpretação crítica dos resultados obtidos.

## 3. Escolha da região de estudo

Cada grupo deverá escolher uma cidade, bairro ou região urbana. Exemplos possíveis:

- Natal/RN;
- Ponta Negra;
- Lagoa Nova;
- Mossoró;
- João Pessoa;
- Recife;
- outra região escolhida pelo grupo.

A escolha deve ser justificada brevemente no `README.md` que acompanhará o repositório do projeto no Github.

>  🚨🚨🚨 Recomenda-se evitar regiões grandes demais, pois isso pode tornar o processamento mais lento e dificultar a interpretação visual.

## 4. Etapa 1 – Construção do grafo com OSMnx

O grupo deverá utilizar a biblioteca **OSMnx** para baixar a rede viária da região escolhida.

A rede deverá ser obtida preferencialmente com:

```python
network_type="drive"
```


O grafo deve representar:

- **nós**: interseções ou pontos relevantes da malha viária;
- **arestas**: vias ou segmentos de ruas.

Cada nó extraído pelo OSMnx possui atributos geográficos importantes:

- `x`: longitude;
- `y`: latitude.

Esses atributos serão usados posteriormente para visualização geográfica no Gephi.

In [5]:
# Exemplo inicial de uso do OSMnx
# Este código é apenas um ponto de partida. O grupo deve adaptar para a região escolhida.

# !pip install osmnx

import osmnx as ox
import networkx as nx

place = "Lagoa Nova, Natal, Rio Grande do Norte, Brazil"

G = ox.graph_from_place(place, network_type="drive")

print(G)
print(f"Número de nós: {G.number_of_nodes()}")
print(f"Número de arestas: {G.number_of_edges()}")

MultiDiGraph with 1189 nodes and 2712 edges
Número de nós: 1189
Número de arestas: 2712


In [6]:
G_proj = ox.project_graph(G)
ox.save_graphml(G_proj, filepath="./lagoa_nova_natal.graphml")

## 5. Etapa 2 – Análise estrutural com NetworkX

Após a construção do grafo, o grupo deverá calcular métricas estruturais usando **NetworkX**.

### Métricas obrigatórias

O trabalho deve conter, **no mínimo**:

1. grau dos nós;
2. distribuição de grau;
3. identificação dos nós com maior grau, isto é, hubs;
4. betweenness centrality;
5. closeness centrality;
6. core number;
7. análise do k-core da rede.

### Atenção

Algumas métricas podem exigir a conversão do grafo para uma versão não direcionada ou simplificada. O grupo deve explicar as escolhas feitas.

In [ ]:
# Exemplo de preparação do grafo para algumas análises

# Conversão para grafo não direcionado, quando necessário
G_undirected = ox.convert.to_undirected(G)

# Grau dos nós
degree_dict = dict(G_undirected.degree())

# Top 10 nós por grau
top_degree = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:10]
top_degree

In [ ]:
# Exemplo de cálculo do core number

core_number = nx.core_number(G_undirected)

# Maior valor de core encontrado
max_core = max(core_number.values())
print(f"Maior core number: {max_core}")

# Nós pertencentes ao núcleo mais denso
main_core_nodes = [node for node, core in core_number.items() if core == max_core]
print(f"Número de nós no núcleo principal: {len(main_core_nodes)}")

## 6. Questões analíticas obrigatórias

O grupo deverá responder explicitamente às seguintes perguntas:

1. Os nós com maior grau coincidem com os nós de maior betweenness?
2. O núcleo identificado pelo k-core coincide com os principais hubs?
3. O que a métrica de betweenness revela que o grau não revela?
4. O que muda quando a rede é analisada em sua posição geográfica real e quando é analisada por um layout estrutural?
5. Existem regiões críticas para mobilidade urbana na área analisada?
6. A rede parece homogênea ou apresenta concentração estrutural?
7. Os resultados obtidos fazem sentido considerando o conhecimento urbano da região escolhida?

Essas respostas devem aparecer no `README.md`, no notebook e no vídeo (sim, a apresentação será assíncrona :-).

## 7. Etapa 3 – Exportação do grafo para o Gephi

O grafo deverá ser exportado no formato `.graphml`, que pode ser importado diretamente no Gephi.

Antes da exportação, recomenda-se adicionar ao grafo os atributos calculados, como grau, betweenness, closeness e core number.

Exemplo:

In [ ]:
# Adicionando atributos ao grafo

nx.set_node_attributes(G_undirected, degree_dict, "degree")
nx.set_node_attributes(G_undirected, core_number, "core_number")

# Atenção: betweenness pode ser custoso em redes grandes.
# Para redes grandes, o grupo pode usar aproximações ou justificar a escolha metodológica.

# betweenness = nx.betweenness_centrality(G_undirected, normalized=True)
# nx.set_node_attributes(G_undirected, betweenness, "betweenness")

ox.save_graphml(G_undirected, "rede_urbana.graphml")

## 8. Etapa 4 – Visualização com Gephi

O grupo deverá utilizar o **Gephi** para produzir visualizações avançadas da rede.

### Plugin obrigatório

Instalar o plugin:

- **Geo Layout** (ou similar)

Esse plugin permite utilizar os atributos geográficos dos nós para posicioná-los no espaço.

No arquivo exportado pelo OSMnx, os atributos são:

- `x`: longitude;
- `y`: latitude.

No Gephi, o grupo deverá configurar o Geo Layout usando:

- longitude → `x`;
- latitude → `y`.

## 9. Visualizações obrigatórias no Gephi

O grupo deverá produzir pelo menos duas perspectivas visuais da rede.

### 9.1 Visualização geográfica

Nesta visualização, a rede deve preservar a forma real da cidade ou bairro.

Requisitos:

- usar o plugin Geo Layout (ou similar);
- posicionar os nós com base em latitude e longitude;
- destacar visualmente as métricas calculadas.

### 9.2 Visualização estrutural

Nesta visualização, o grupo deverá aplicar um layout de força.

Layout obrigatório:

- ForceAtlas2.

Layouts opcionais:

- Fruchterman-Reingold;
- Yifan Hu.

Essa visualização não precisa representar a geografia real da cidade. O objetivo é revelar a organização estrutural do grafo.

## 10. Codificação visual obrigatória

Nas visualizações no Gephi, o grupo deverá utilizar atributos visuais para facilitar a interpretação.

Requisitos mínimos:

- tamanho do nó proporcional ao grau;
- cor do nó associada ao core number;
- destaque dos nós com maior betweenness;
- visualização do subgrafo correspondente ao k-core escolhido.

O grupo deve explicar o significado de cada escolha visual.

## 11. Filtros obrigatórios no Gephi

O grupo deverá aplicar e analisar pelo menos os seguintes filtros:

1. **Top 10% dos nós por grau**;
2. **Subgrafo com k-core maior ou igual a um valor k**, escolhido e justificado pelo grupo.

A escolha do valor de `k` deve ser explicada. Não basta aplicar o filtro; é necessário interpretar o que ele revela sobre a estrutura urbana.

## 12. Entregáveis

Ao final, cada grupo deverá entregar dois itens principais.

### 12.1 Repositório no GitHub

O grupo deverá submeter um repositório no GitHub contendo tudo o que foi gerado.

O repositório deve conter, no mínimo:

- notebook `.ipynb` funcional;
- códigos utilizados;
- arquivo `.graphml` exportado;
- imagens produzidas no Python e/ou Gephi;
- arquivos auxiliares, quando houver;
- `README.md` com a descrição do trabalho.

O `README.md` deve conter:

- identificação da região analisada;
- link do video na plataforma Loom;
- objetivo do trabalho;
- metodologia;
- métricas calculadas;
- principais visualizações;
- respostas às questões obrigatórias;
- principais conclusões.

### 12.2 Vídeo de apresentação

Cada grupo deverá produzir um vídeo de até **15 minutos** apresentando o trabalho.

O vídeo deve conter:

0. Apresentação do Grupo e como as tarefas foram divididas entre todos os membros;
1. apresentação da região escolhida;
2. explicação da construção do grafo;
3. apresentação das métricas calculadas;
4. visualização geográfica no Gephi;
5. visualização estrutural no Gephi;
6. análise dos hubs;
7. análise do k-core;
8. discussão crítica dos resultados;
9. conclusões do grupo.

O vídeo deve ser objetivo e analítico. Não deve ser apenas uma execução de código. Utilizar a Ferramenta Loom para a gravação do video e inserir o link no repositório do Github. Caso seja necessário, os grupos podem quebrar o video e mais de um, porém, a duração somada não poderá ultrapassar os 15min.

## 13. Critérios de avaliação

| Critério | Peso |
|---|---:|
| Construção correta do grafo com OSMnx | 15% |
| Aplicação adequada das métricas de grafos | 20% |
| Uso correto e interpretação do k-core | 20% |
| Qualidade das visualizações no Gephi | 20% |
| Interpretação crítica dos resultados | 15% |
| Organização do repositório e qualidade do vídeo | 10% |

A avaliação considerará tanto a implementação quanto a capacidade do grupo de interpretar os resultados obtidos. **A não entrega do Video na plataforma Loom irá impactar nota zero ao trabalho**.

## 14. Observações finais

Este trabalho conecta conceitos fundamentais de Estrutura de Dados II com uma aplicação real em redes urbanas.

A expectativa é que os alunos compreendam que grafos não são apenas uma estrutura abstrata, mas uma forma poderosa de representar, analisar e interpretar sistemas complexos do mundo real.

A qualidade do trabalho será medida não apenas pelo código, mas principalmente pela capacidade de transformar métricas de grafos em argumentos interpretáveis sobre a estrutura da cidade analisada.

A nota final irá compor 100% da Unidade 2.